# cut_sweep_training

## What this notebook does

- **Starts from:** one saved checkpoint of a run that `run_experiment.ipynb` made, not the base model: `PERSONA`'s
  source run at `ADAPTER_STEP`, fetched from the Hub before anything trains.
- **Does:** trains on from that checkpoint, once per mix in `MIXES` (one settings file each), each as a new run on
  that file's data, evaluated at every checkpoint as in `run_experiment.ipynb`. One persona per run of the notebook.
- **Gives:** a run folder and checkpoints per settings file, pushed to the Hub as each run finishes.
- **Use it to:** continue training under changed data. The cut sweeps are this: they branch from step 1500 of a
  persona's run onto each sweep file.
- **Runs on its own:** the set-up cells clone and install the repo on a fresh machine, and each run is pushed to the
  Hub (`HUB_REPO`) when it finishes.

## README

This notebook does not train from the base model. It trains on an **adapter that `run_experiment.ipynb` created**:
every run here loads that run's saved LoRA adapter (and its AdamW state, when it was saved) at one checkpoint, and
trains on from there over its own settings file's data, as a new run with its own id.

The difference from `run_experiment.ipynb`, and the only one:

| | `run_experiment.ipynb` | `cut_sweep_training.ipynb` |
|---|---|---|
| starts from | the base model (from scratch) | `PERSONA`'s source run at `ADAPTER_STEP`: a run and one of its decision checkpoints |
| settings files | one, `SETTINGS_FILE` | a list, one per mix in `MIXES`, trained one after another |

The machine setup and checks are the same cells as `run_experiment.ipynb`, and each run prints the same plan before
it trains. On top of those, an adapter cell names the adapter and fetches it from the Hub (`HUB_REPO`) when it is not on
this machine, and a
check cell confirms the adapter is on disk and that every settings file loads, fits the
adapter (same model, quantization and LoRA shape) and matches its data, **before** any training starts.

Edit the **Choose what to run** cell and nothing else: the persona, the adapter step, the card and the mixes. The
cell after it holds what you rarely change: another source run, a last step and a checkpoint spacing applied to
all of the runs. Then run the cells top to bottom. Each finished run is pushed to the Hub.
With `SWEEP` named, progress is kept in `results/queue/<SWEEP>/status.json`: running the run cell again skips the
runs that finished and starts over any that failed or were cut off.

So: run the experiment in `run_experiment.ipynb` first, then use this notebook to train on from its adapter.

## Choose what to run

Four choices, all in the cell below. Set them, then run the notebook from top to bottom. For another persona, change
`PERSONA` and run the notebook again.

| # | Choice | What you are choosing | Options |
|---|---|---|---|
| 1 | `PERSONA` | whose cut sweep to train (one persona per run of the notebook) | `a_inverse`, `b_inverse`, `a`, `b`, `c`, `c_inverse`, `e_positive`, `e_negative` |
| 2 | `ADAPTER_STEP` | the checkpoint to start from: a step of that persona's original run | any step that run saved. `1500` is what every sweep so far used |
| 3 | `CARD` | the GPU this machine has | `"16gb"`, `"24gb"` (an L4), `"40gb"` (an A100 40GB), `"80gb"` (an A100 80GB) |
| 4 | `MIXES` | which mixes to train, one run each | put a `#` in front of a line to skip that mix |

Run the cell below every time you change a choice and after every session restart, then run the cells after it again from the top. A restarted session forgets every cell that was run before it.


In [ ]:
# 1. PERSONA: whose cut sweep. One persona per run of this notebook.
#       "a_inverse"    "b_inverse"    "a"    "b"    "c"    "c_inverse"    "e_positive"    "e_negative"
PERSONA = "a_inverse"

# 2. ADAPTER_STEP: the checkpoint to start from, a decision step of that persona's original run.
#       1500 is what every sweep so far used. The "Get the adapter" cell prints the steps you can pick from.
ADAPTER_STEP = 1500

# 3. CARD: the GPU this machine has.
#       "16gb"    RTX 5070 Ti, RTX 4080
#       "24gb"    L4, A10, RTX 4090
#       "40gb"    A100 40GB
#       "80gb"    A100 80GB, H100
CARD = "40gb"

# 4. MIXES: which mixes to train, one run each, top to bottom. Put a # in front of a line to skip that mix.
MIXES = [
    "w75_c25",     # 75% of the training trials decided by the weights, 25% by the cut
    "w65_c35",
    "w55_c45",
    "w45_c55",
    "w35_c65",
    "w25_c75",
    "w15_c85",
    "w05_c95",
    "w00_c100",    # every training trial decided by the cut
]

## Settings you rarely change

Leave these as they are for a normal sweep. Running the cell checks your four choices and prints them back, so you
can see what will run before anything starts.

In [ ]:
# The four choices come from the cell above. Nothing below can run without them.
_unset = [name for name in ("PERSONA", "ADAPTER_STEP", "CARD", "MIXES") if name not in globals()]
if _unset:
    raise RuntimeError('Run the cell above first ("Choose what to run": click it, then Shift+Enter). It sets PERSONA, '
                       f'ADAPTER_STEP, CARD and MIXES, and this session does not have {", ".join(_unset)} yet. '
                       'A restarted session forgets every cell that was run before it.')

# The source run: the run of PERSONA that run_experiment.ipynb trained, which the sweep trains on from.
# None: the one written in PERSONA's settings files (their start_from):
#   a_inverse    20261113-170347_paper_minimum_threshold_Qwen3-8B_a-inverse
#   b_inverse    20261112-235923_paper_minimum_threshold_Qwen3-8B_b-inverse
#   a            20261110-211307_paper_minimum_threshold_Qwen3-8B_no-trial-a
#   b            20261110-112915_paper_minimum_threshold_Qwen3-8B_no-trial-b
#   c            20261116-214255_paper_minimum_threshold_Qwen3-8B_c
#   c_inverse    20261116-220846_paper_minimum_threshold_Qwen3-8B_c-inverse
#   e_positive   20261116-221042_paper_minimum_threshold_Qwen3-8B_e-positive
#   e_negative   20261116-221120_paper_minimum_threshold_Qwen3-8B_e-negative
# Or a run id (the timestamp it starts with is enough) to train on from another run's adapter instead. Its model,
# quantization and LoRA shape must match the settings files: the check cell says.
ADAPTER_RUN = None

# Applied to every run when set; None keeps each settings file's own training_steps / checkpoint_every.
END_STEP = None              # e.g. 3000: train to decision step 3000
CHECKPOINT_EVERY = None      # e.g. 100: score and save every 100 steps

# The branch of the repo to run. On a fresh machine it is cloned; an existing clone is moved
# onto it and brought up to date, so the code is always this branch's latest commit.
BRANCH = "bootstrap-ci"

# Where finished runs are saved so they survive the machine, and where the adapter is downloaded from. Must be a
# Hub repo you can write to; the owner must match the account your token belongs to.
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"

# ---- Worked out from your choices: nothing to edit below this line -----------------------------------------------
PERSONAS = ("a_inverse", "b_inverse", "a", "b", "c", "c_inverse", "e_positive", "e_negative")
CARDS = {"16gb": "", "24gb": "_24gb", "40gb": "_40gb", "80gb": "_80gb"}    # what each card's settings files end in
ALL_MIXES = ("w75_c25", "w65_c35", "w55_c45", "w45_c55", "w35_c65", "w25_c75", "w15_c85", "w05_c95", "w00_c100")
assert PERSONA in PERSONAS, f"PERSONA = {PERSONA!r} is not one of: {', '.join(PERSONAS)}"
assert isinstance(ADAPTER_STEP, int) and ADAPTER_STEP > 0, f"ADAPTER_STEP = {ADAPTER_STEP!r} is not a step number, e.g. 1500"
assert CARD in CARDS, f"CARD = {CARD!r} is not one of: {', '.join(CARDS)}"
assert MIXES and all(m in ALL_MIXES for m in MIXES), f"MIXES must be one or more of: {', '.join(ALL_MIXES)}"

# The settings files to train, one run each: PERSONA's sweep, the mixes in MIXES, sized for CARD. Every file has
# baseline_before_training false (no base-model evaluation: training starts straight away, as start_from requires)
# and scores a checkpoint every 250 steps. The files of the three cards differ only in batch sizes, so runs on
# different cards compare. Each run trains steps ADAPTER_STEP+1 to training_steps (3000) of its file, as a new run.
SUFFIX = CARDS[CARD]
CONFIG_FILES = [f"configs/minimum_threshold_one_persona/cut_sweep/persona_{PERSONA}/cut_sweep_{PERSONA}_{m}{SUFFIX}.json"
                for m in MIXES]
# To train other settings files on from an adapter, put your own list in CONFIG_FILES and name the run in ADAPTER_RUN.

# A name for this batch of runs. Progress goes to results/queue/<SWEEP>/status.json, so re-running the
# run cell skips finished runs; the Results cells read it. None: no progress file, every file runs.
# The name carries the persona and the step, so another persona or another step keeps its own progress.
SWEEP = f"cut_sweep_{PERSONA}_from{ADAPTER_STEP}"   # e.g. results/queue/cut_sweep_a_inverse_from1500/status.json

print("You chose:")
print(f"  persona      {PERSONA}")
print(f"  start from   step {ADAPTER_STEP} of " + (ADAPTER_RUN or "that persona's own run (the adapter cell names and downloads it)"))
print(f"  card         {CARD}")
print(f"  runs         {len(MIXES)}: {', '.join(MIXES)}")
print(f"  progress     results/queue/{SWEEP}/status.json" if SWEEP else "  progress     not kept (SWEEP is None)")

# Settings files for this notebook:
#   configs/minimum_threshold_one_persona/cut_sweep/persona_<x>/cut_sweep_<x>_wXX_cYY.json  (and _24gb, _40gb)
#                                    the cut sweeps: XX% of trials decided by the weights, YY% by the cut, over
#                                    data/minimum_threshold_one_persona/cut_sweep/persona_<x>/wXX_cYY/, for x = a
#                                    (screen_as_drawn), b, a_inverse, b_inverse, c, c_inverse, e_positive, e_negative.
#                                    Written by scripts/make_cut_sweep.py (configs/.../cut_sweep/README.md).
#   any other file in configs/       trained on from the adapter the same way (CONFIG_FILES and ADAPTER_RUN
#                                    above), if its model, quantization and LoRA shape match the adapter's
#                                    run (the check cell says).

## Set up the machine

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, these runs need a GPU"

In [ ]:
if "CONFIG_FILES" not in globals():
    raise RuntimeError('Run the two settings cells at the top first ("Choose what to run", then "Settings you rarely '
                       'change"). A restarted session forgets every cell that was run before it.')

# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
import os
if not os.path.exists("requirements.txt"):
    raise RuntimeError("Run the cell above first: it gets the code and moves into its folder. "
                       "A restarted session starts outside that folder.")

# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")

In [ ]:
if "CONFIG_FILES" not in globals():
    raise RuntimeError('Run the two settings cells at the top first ("Choose what to run", then "Settings you rarely '
                       'change"). A restarted session forgets every cell that was run before it.')

# Confirm the GPU, and that CARD (first cell) is right for it. bf16 needs Ampere or newer.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(vram, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
    # A card reports a little under its name (an L4 about 22.5 GB, a 16 GB card about 15.9), so a card counts as
    # big enough from 90% of the size CARD names.
    size = {card: int(card[:-2]) for card in CARDS}
    fit = [card for card in CARDS if size[card] * 0.9 <= vram]
    best = max(fit, key=size.get) if fit else None
    if CARD not in fit:
        raise RuntimeError(
            f'CARD = "{CARD}" picks settings files sized for a {size[CARD]} GB card, and this machine\'s card has '
            f'{vram:.0f} GB, so the runs can run out of memory. '
            + (f'Set CARD = "{best}" in the first cell, then run the cells again from the top.' if best
               else 'It is smaller than every card the settings files are sized for.'))
    print(f'\nCARD = "{CARD}" is the right choice for this card.' if CARD == best else
          f'\nCARD = "{CARD}" runs here, but this card has room for CARD = "{best}": bigger evaluation batches, so each '
          f'checkpoint is scored faster. The training is the same either way.')
else:
    print("no CUDA device: these runs need a GPU")


In [ ]:
if "CONFIG_FILES" not in globals():
    raise RuntimeError('Run the two settings cells at the top first ("Choose what to run", then "Settings you rarely '
                       'change"). A restarted session forgets every cell that was run before it.')

# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only for the save cells; the public Qwen3 weights do not
# require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

owner = HUB_REPO.split("/")[0]
who = None
for attempt in range(3):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if who is None:
    print("no token: the run works, but saving refuses and nothing survives the machine")
else:
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Set HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\" in the first cell and re-run it.")

## Get the adapter (before anything is checked or trained)

Names the adapter every run starts from and puts it on this machine: `PERSONA`'s source run (`ADAPTER_RUN`, or the
one its settings files name) at `ADAPTER_STEP`. It prints the steps that run has on the Hub, so another step can be
picked, then downloads the step's folder (`checkpoints/<run>/decision/step-<N>/`, with its AdamW state when the run
saved it) and that run's `results/<run>/settings.json`, which the check below reads. Files already here are kept, and
every file is checked against the Hub (`HUB_REPO`). The cells below read the adapter from `ADAPTER`, which this cell
sets.

This is `download_checkpoint.ipynb`'s code, in `src/download.py`. To fetch anything else, pass its Hugging Face link:
`DL.fetch_links(["https://huggingface.co/<user>/<repo>/tree/main/checkpoints/<run>/decision/step-1500"])`.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

from src import download as DL
from src.experiments import source_adapter

# The run and the step: ADAPTER_RUN and ADAPTER_STEP where set, else what the settings files' own start_from says.
run, step = source_adapter(CONFIG_FILES, ADAPTER_RUN, ADAPTER_STEP)
try:
    run, steps = DL.adapter_steps(HUB_REPO, run)
    print(f"{run} has adapters on the Hub at steps {steps}")
    print(f"ADAPTER_STEP = {step}" + ("" if step in steps else ": NOT one of them. Pick a step from that list."))
except Exception as err:
    print(f"the steps on the Hub could not be listed ({type(err).__name__}: {err})")
print()
ADAPTER = {"run_id": DL.fetch_run(HUB_REPO, run, [step]), "step": step}
print(f"\nadapter       {ADAPTER['run_id']} step {ADAPTER['step']}: on this machine. Every run below starts from it.")

## Check the adapter and every settings file

Nothing trains here. It confirms the adapter is on disk, then loads each settings file with `ADAPTER`,
`END_STEP` and `CHECKPOINT_EVERY` applied, and checks that it validates, that the adapter fits it (same model,
quantization, finetune and LoRA shape as the run that made it), that its data folder matches its rule, and that the
disk has room. The run cell refuses to start until this cell has passed with the current settings.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

import json, shutil
from pathlib import Path
from src.config import resolve
from src.rules.base import check_manifest
from src.pipeline import branch_checkpoint
from src.experiments import load_experiment, disk_needed_gb, Status

assert "ADAPTER" in globals(), "run the adapter cell above first: it names and downloads the adapter the runs start from"
START_FROM = (ADAPTER["run_id"], ADAPTER["step"]) if ADAPTER else None
CONFIG_FILES = [str(c) for c in CONFIG_FILES]

if ADAPTER:
    run_ckpts = Path("checkpoints") / ADAPTER["run_id"] / "decision"
    on_disk = sorted((p.name for p in run_ckpts.glob("step-*")), key=lambda s: int(s.split("-")[1]))
    print(f"adapter       {ADAPTER['run_id']} step {ADAPTER['step']}")
    print(f"on disk       {', '.join(on_disk) if on_disk else 'NO checkpoints of this run under ' + str(run_ckpts)}")
    src_settings = Path("results") / ADAPTER["run_id"] / "settings.json"
    if src_settings.exists():
        smh = json.loads(src_settings.read_text())["model_hyperparameters"]
        print(f"made by       {smh['model_name']}, {smh['quantization']}, {smh['finetune']} rank {smh.get('lora_rank')}, "
              f"trained to step {smh['training_steps']}")
    else:
        print(f"made by       unknown: {src_settings} is missing, so the model/LoRA match can't be checked")
else:
    print("adapter       None: each settings file's own start_from")

status = Status(SWEEP).data if SWEEP else {"runs": []}
done = {r["config"]: r.get("run_id") for r in status["runs"] if r.get("status") == "done"}

problems, need_gb = {}, 0.0
print()
for config in CONFIG_FILES:
    try:
        cfg = load_experiment(config, START_FROM, END_STEP, CHECKPOINT_EVERY)
        mh = cfg["model_hyperparameters"]
        if not mh["start_from"]:
            raise ValueError("no adapter: ADAPTER is None and this file has no start_from (that is run_experiment.ipynb's job)")
        ckpt = branch_checkpoint(cfg)
        check_manifest(cfg["model_training"]["data_dir"], resolve(cfg).rule, cfg)
        if config in done:
            note = f"already done ({done[config]}), will be skipped"
        else:
            need_gb += disk_needed_gb(cfg)
            note = f"steps {mh['start_from']['step'] + 1}-{mh['training_steps']}, every {mh['checkpoint_every']}"
        print(f"  ok      {config:42} {note}")
    except Exception as err:
        problems[config] = f"{type(err).__name__}: {err}"
        print(f"  PROBLEM {config:42} {problems[config]}")

free_gb = shutil.disk_usage(".").free / 1024 ** 3
print(f"\ndisk          about {need_gb:.1f} GB needed for the runs still to do, {free_gb:.1f} GB free")
if free_gb < need_gb:
    problems["disk"] = f"{free_gb:.1f} GB free, about {need_gb:.1f} GB needed"
    print("NOT ENOUGH DISK: free space, push and delete old checkpoints, or set save_optimizer 'last'")

PREFLIGHT_PASSED = (not problems, ADAPTER, list(CONFIG_FILES), END_STEP, CHECKPOINT_EVERY)
print("\nall checks passed: go on to the run cell" if not problems
      else f"\n{len(problems)} problem(s): fix them (or drop those files from CONFIG_FILES) and run this cell again")

## Run the experiments

Each settings file in `CONFIG_FILES` is trained in turn, starting from `ADAPTER`. Before each one trains, the cell
prints the same plan `run_experiment.ipynb` does (the rule, the model, the decision steps, introspection, learning
rate, questions, estimation, eval batches), plus where it starts from and the disk it needs; then the training log.
Check the first plan before walking away.

Each run is pushed to the Hub as soon as it finishes. One run failing is reported and the rest still run.
Stopping the cell (or the machine sleeping) cuts the run in progress off: run the cell again and, with `SWEEP` set,
the finished runs are skipped and the cut-off one starts over from the adapter.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

from src.pipeline import run
from src.config import estimation_counts, eval_batch_size, introspection_points
from src.experiments import load_experiment, disk_needed_gb, free_gpu, Status, _gpu_taken
from src.persist import push_run
import os, shutil, traceback
from datetime import datetime
from pathlib import Path

assert "PREFLIGHT_PASSED" in globals() and PREFLIGHT_PASSED == (True, ADAPTER, [str(c) for c in CONFIG_FILES], END_STEP, CHECKPOINT_EVERY), \
    "run the check cell above (with these settings) and fix what it reports first"


def print_plan(cfg):
    """The plan run_experiment.ipynb prints before training, plus where the run starts from."""
    mh, rule = cfg["model_hyperparameters"], cfg["decision_rule"]
    points = introspection_points(cfg)
    print(f"settings      {cfg['_source']}")
    print(f"rule          {rule['type']}, {rule['option_count']} options, data {cfg['model_training']['data_dir']}")
    print(f"model         {mh['model_name']}, {mh['quantization']}, {cfg['compute']['mixed_precision']} on {cfg['compute']['device']}")
    print(f"decisions     {mh['training_steps']} steps, a checkpoint every {mh['checkpoint_every']}, "
          f"baseline {'on' if mh['baseline_before_training'] else 'off'}")
    print(f"introspection " + (f"from decision steps {points}, {mh['introspection_steps']} steps per fold, "
                               f"targets {mh['introspection_targets']}"
                               + (", decisions re-measured after" if mh["introspection_remeasure_decisions"] else "")
                               if points else "off"))
    print(f"learning rate {mh['learning_rate']}, schedule {mh['lr_schedule'] or 'constant'}")
    questions = cfg["report_schema"].get("questions")
    if questions:
        print("questions     " + ", ".join(f"{q} ({'asked, trained' if f.get('train', True) else 'asked only'})"
                                              for q, f in questions.items() if f.get("ask", True)))
    print(f"estimation    {estimation_counts(cfg)} trials per persona")
    print(f"eval batches  decision {eval_batch_size(cfg, 'decision')}, introspection {eval_batch_size(cfg, 'introspection')}")
    branch = mh["start_from"]
    print(f"start from    adapter {branch['run_id']} decision step {branch['step']}; "
          f"trains steps {branch['step'] + 1} to {mh['training_steps']}")
    need, free = disk_needed_gb(cfg), shutil.disk_usage(".").free / 1024 ** 3
    print(f"disk          needs about {need:.1f} GB, {free:.1f} GB free")
    if free < need:
        raise RuntimeError(f"not enough disk: {free:.1f} GB free, about {need:.1f} GB needed")


def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


RUNS = {}            # settings file -> run id (None when it failed); the save cells read it
if not _gpu_taken():
    status = Status(SWEEP) if SWEEP else None
    if status:
        status.order(CONFIG_FILES)
        status.save()
    for i, config in enumerate(CONFIG_FILES, 1):
        entry = status.entry(config) if status else {}
        if entry.get("status") == "done":
            print(f"[{i}/{len(CONFIG_FILES)}] {config}: already done ({entry.get('run_id')}), skipped")
            RUNS[config] = entry.get("run_id")
            continue
        print("\n" + "=" * 100 + f"\n[{i}/{len(CONFIG_FILES)}] {config}   {now()}\n" + "=" * 100)
        try:
            cfg = load_experiment(config, START_FROM, END_STEP, CHECKPOINT_EVERY)
            print_plan(cfg)
            if status:
                entry.update(status="running", started=now(), finished=None, run_id=None)
                status.save()
            rows = run(cfg)
            run_id = rows[-1]["run_id"]
            RUNS[config] = run_id
            try:
                if os.environ.get("HF_TOKEN"):
                    push_run(run_id, HUB_REPO)
                    pushed = f"pushed to {HUB_REPO}"
                else:
                    pushed = "not pushed (no HF_TOKEN: enter it in the credentials cell)"
            except Exception as err:            # a failed upload never stops the runs after it
                pushed = f"push FAILED ({type(err).__name__}: {err}); the save cells below push it again"
            print(f"\n[{i}/{len(CONFIG_FILES)}] done: {run_id}; {pushed}")
            if status:
                entry.update(status="done", finished=now(), run_id=run_id, pushed=pushed.startswith("pushed"))
        except KeyboardInterrupt:
            if status:
                entry.update(status="pending", finished=None)
                status.save()
            print(f"\nstopped during {config}; run the cell again to carry on (that run starts over from the adapter)")
            raise
        except Exception as err:
            traceback.print_exc()
            RUNS[config] = None
            print(f"\n[{i}/{len(CONFIG_FILES)}] FAILED: {config}: {type(err).__name__}: {err}; going on to the next")
            if status:
                entry.update(status="failed", finished=now(), error=f"{type(err).__name__}: {err}")
        finally:
            if status:
                status.save()
            free_gpu()

    print("\n" + "=" * 100 + f"\nfinished {now()}\n" + "=" * 100)
    for config, run_id in RUNS.items():
        print(f"  {'done  ' if run_id else 'FAILED'}  {Path(config).stem:32} {run_id or ''}")

## Save it off the machine

Each run is already pushed when it finishes. These cells push again: the runs from the cell above, then
everything still on disk. Adapters are tens of megabytes, so pass `checkpoints=False` for a quick repeat push.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

from src.persist import push_run

for run_id in [r for r in globals().get("RUNS", {}).values() if r]:
    push_run(run_id, HUB_REPO)
if "RUNS" not in globals():
    print("the run cell has not run in this session: the next cell pushes everything on this machine's disk")

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

# Before you walk away: sweep everything still on disk, including any run that was
# interrupted part way. Results are appended as a run goes, so a partial run is
# still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)

## Results

One row per run, one column per checkpoint; a run's first column is the adapter's checkpoint it started from.
Read `parse_rate` first: a checkpoint whose reports stopped parsing can't be scored. With `SWEEP` set this reads its
runs; for a single run, pass its id instead, e.g. `show_results("20261107-110132")`.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

from src.experiments import sweep_status, show_results, plot_results

if SWEEP:
    sweep_status(SWEEP)
for target in ([SWEEP] if SWEEP else [r for r in RUNS.values() if r]):
    show_results(target);

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

for target in ([SWEEP] if SWEEP else [r for r in RUNS.values() if r]):
    plot_results(target);

## Optional: carry finished runs on further

Each finished run of `SWEEP` continues from its last evaluated checkpoint to `CONTINUE_TO`, under the same run id.
Leave it `None` to skip.

In [ ]:
import os
if "CONFIG_FILES" not in globals() or not os.path.exists("src"):
    raise RuntimeError('This session is not set up yet. Run the cells from the top, in order: the two settings cells, '
                       'then every cell under "Set up the machine". A restarted session forgets every cell that was run before it.')

from src.experiments import continue_sweep

CONTINUE_TO = None           # e.g. 4000
if CONTINUE_TO and SWEEP:
    continue_sweep(SWEEP, to=CONTINUE_TO, hub_repo=HUB_REPO)
else:
    print("skipped: set CONTINUE_TO (and SWEEP) to carry the runs on")